# Compare Black Friday week year over year

*Data Analysis in Python · Lesson 6 final exercise*

The e-commerce team wants to know how Black Friday week 2025 did against the same week of 2024, by week and by day, before the Monday meeting.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

Add your cells between the cell that builds the `sessions` table and the **Check your work** cell. If you're stuck on a step, open its hint under the **Check your work** cell.

1. Create the `daily` table: one row per date of the `sessions` table, with the dates as the index, and the columns `sessions`, `transactions` and `revenue`.
2. Create the `weekly` table: the `daily` table totaled by Monday-to-Sunday week, with only the weeks that have all 7 days, and two more columns: `sessions_yoy` and `revenue_yoy`, each week's percent change from the same week a year earlier, as fractions such as `0.12` for +12%.
3. Create `revenue_7d`: the 7-day rolling average of the `daily` table's revenue, as a Series.
4. Create the `bf_week` table: the rows of the `daily` table for Black Friday week 2025, from the Monday before `BLACK_FRIDAY` to the Sunday after it, with the columns `sessions`, `transactions` and `revenue`. Then create the `last_year_week` table: the same week of 2024, with the same weekdays.
5. Create `bf_week_yoy`: the `bf_week` table's totals over the `last_year_week` table's, minus 1, as a Series with `sessions`, `transactions` and `revenue`.
6. Add a text cell for the e-commerce team: how much did Black Friday week grow on 2024, which days of 2024 is it compared with, and why those? Give the numbers, and the 7-day average of revenue on Black Friday itself, both years.
7. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: The `daily` table has 457 rows, October 1, 2024 to December 31, 2025. The `weekly` table has the 64 full weeks, from the week ending October 13, 2024 to the one ending December 28, 2025, and in the week ending November 30, 2025, `revenue_yoy` is about `0.4312` and `sessions_yoy` about `0.4058`. `revenue_7d` is about 21,465.39 on Black Friday 2025 and 14,926.76 on Black Friday 2024. The `bf_week` table is November 24 to 30, 2025, with 64,194 sessions and $161,833.46 of revenue, and the `last_year_week` table November 25 to December 1, 2024, with 45,664 sessions and $113,072.02. `bf_week_yoy` is sessions `0.4058`, transactions `0.4765` and revenue `0.4312`.

## Parameters

In [ ]:
import pandas as pd

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
MONTHS = [f"{year}-{month:02d}" for year in (2024, 2025) for month in range(1, 13)][9:]  # October 2024 to December 2025
START, END = "2024-10-01", "2025-12-31"
BLACK_FRIDAY = pd.Timestamp("2025-11-28")

In [ ]:
# The clean sessions table, with lesson 3's steps. Run it, and leave it as it is.
DAY_FIRST_MONTHS = ["2025-03"]
SOURCE_ALIASES = {
    "facebook.com": "facebook", "m.facebook.com": "facebook", "l.facebook.com": "facebook",
    "lm.facebook.com": "facebook", "fb": "facebook",
    "instagram.com": "instagram", "l.instagram.com": "instagram", "ig": "instagram",
}


def read_export(month):
    """Read one month's export with dates as dates, revenue as numbers and lost campaigns as missing."""
    export = pd.read_csv(DATA_URL + f"sessions_export_{month.replace('-', '_')}.csv",
                         na_values={"session_campaign": ["(not set)", "-"]})
    date_format = "%d/%m/%Y" if month in DAY_FIRST_MONTHS else "%Y-%m-%d"
    export["date"] = pd.to_datetime(export["date"], format=date_format)
    if not pd.api.types.is_numeric_dtype(export["revenue"]):
        export["revenue"] = export["revenue"].str.replace("$", "").str.replace(",", "").astype(float)
    return export


raw = pd.concat([read_export(month) for month in MONTHS], ignore_index=True)
raw["session_campaign"] = raw["session_campaign"].fillna("(not set)")
deduped = raw.drop_duplicates()
sessions = deduped[deduped["date"].between(START, END) & (deduped["session_medium"] != "test")].reset_index(drop=True)
sessions["source"] = sessions["session_source"].str.lower().str.strip().replace(SOURCE_ALIASES)
sessions.shape

In [ ]:
daily = sessions.groupby("date")[["sessions", "transactions", "revenue"]].sum()
daily.shape

In [ ]:
days_per_week = daily["sessions"].resample("W").count()
weekly = daily.resample("W").sum()
weekly = weekly[days_per_week == 7].copy()
assert (weekly.index.to_series().diff().dropna() == pd.Timedelta(weeks=1)).all()
weekly["sessions_yoy"] = weekly["sessions"].pct_change(52)
weekly["revenue_yoy"] = weekly["revenue"].pct_change(52)
weekly.loc["2025-10":"2025-12"]

In [ ]:
revenue_7d = daily["revenue"].rolling(7).mean()
revenue_7d.loc[[BLACK_FRIDAY, BLACK_FRIDAY - pd.Timedelta(weeks=52)]]

In [ ]:
bf_week = daily.loc[BLACK_FRIDAY - pd.Timedelta(days=4):BLACK_FRIDAY + pd.Timedelta(days=2)]
last_year = BLACK_FRIDAY - pd.Timedelta(weeks=52)
last_year_week = daily.loc[last_year - pd.Timedelta(days=4):last_year + pd.Timedelta(days=2)]
assert list(bf_week.index.day_name()) == list(last_year_week.index.day_name())
bf_week_yoy = bf_week.sum() / last_year_week.sum() - 1
last_year.day_name(), last_year_week.index[0], last_year_week.index[-1], bf_week_yoy

**For the e-commerce team.** Black Friday week 2025, Monday, November 24 to Sunday, November 30, had 64,194 sessions and $161,833.46 of revenue: up 40.6% and 43.1% on the same week of 2024, and transactions up 47.7%.

It's compared with November 25 to December 1, 2024, the week of Black Friday 2024, which was November 29. Black Friday is set by the weekday, so last year's week is 52 weeks back, Monday to Sunday. The same dates a year back, November 24 to 30, 2024, would start on a Sunday, leave out the Sunday after Black Friday, and show revenue up 54.1%: 11 points of growth that the calendar made up.

On Black Friday itself, the 7-day average of revenue was $21,465.39, against $14,926.76 on Black Friday 2024, up 43.8%. October to December as a whole grew 47.1%, so Black Friday week grew with the business, not ahead of it.

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    black_friday, last_black_friday = pd.Timestamp("2025-11-28"), pd.Timestamp("2024-11-29")
    week = pd.Timestamp("2025-11-30")
    daily = found.get("daily")
    weekly = found.get("weekly")
    if not isinstance(daily, pd.DataFrame) or not isinstance(daily.index, pd.DatetimeIndex):
        results.append((False, "daily should be a DataFrame with the dates as its index: sessions.groupby(\"date\")[[\"sessions\", \"transactions\", \"revenue\"]].sum() (topic 6.1)."))
    elif len(daily) != 457 or not {"sessions", "transactions", "revenue"} <= set(daily.columns):
        results.append((False, f"The daily table should have 457 rows, October 1, 2024 to December 31, 2025, and the columns sessions, transactions and revenue. It has {len(daily):,} rows and {list(daily.columns)}."))
    elif not isinstance(weekly, pd.DataFrame) or not isinstance(weekly.index, pd.DatetimeIndex):
        results.append((False, "weekly should be the daily table totaled by week: daily.resample(\"W\").sum(), with only the full weeks (topic 6.2)."))
    elif len(weekly) == 66:
        results.append((False, "The weekly table has 66 weeks: the partial ones are still in it. The first has 6 days and the last 3, so keep the weeks with 7 (topic 6.2)."))
    elif len(weekly) != 64 or weekly.index[0] != pd.Timestamp("2024-10-13") or week not in weekly.index:
        results.append((False, f"The weekly table should have the 64 full weeks, from the week ending October 13, 2024 to the one ending December 28, 2025. It has {len(weekly)} rows."))
    elif not {"sessions_yoy", "revenue_yoy"} <= set(weekly.columns):
        results.append((False, f"The weekly table should have the columns sessions_yoy and revenue_yoy. It has {list(weekly.columns)}."))
    elif abs(weekly.loc[week, "revenue_yoy"]) > 5 or abs(weekly.loc[week, "sessions_yoy"]) > 5:
        results.append((False, f"The weekly table's revenue_yoy is {weekly.loc[week, 'revenue_yoy']:,.2f} in the week ending November 30, 2025: that's the difference in dollars, diff(52). The brief asks for the percent change: pct_change(52)."))
    elif abs(weekly.loc[week, "revenue_yoy"] - 1.431242) < 0.0001:
        results.append((False, "The weekly table's revenue_yoy is a ratio, 1.43, without the minus 1. pct_change(52) gives the change: 0.43."))
    elif abs(weekly.loc[week, "revenue_yoy"] - 0.862537) < 0.0001:
        results.append((False, "The weekly table's revenue_yoy compares with the week 53 weeks earlier. A year of weeks is 52: pct_change(52)."))
    elif abs(weekly.loc[week, "revenue_yoy"] - 0.431242) > 0.0001 or abs(weekly.loc[week, "sessions_yoy"] - 0.405790) > 0.0001:
        results.append((False, f"In the week ending November 30, 2025, revenue_yoy should be about 0.4312 and sessions_yoy about 0.4058: each week against the row 52 weeks earlier, pct_change(52). They are {weekly.loc[week, 'revenue_yoy']!r} and {weekly.loc[week, 'sessions_yoy']!r}."))
    else:
        results.append((True, "The weekly table has 64 full weeks, and Black Friday week 2025's revenue is up 43.1% on the same week of 2024."))

    trend = found.get("revenue_7d")
    if not isinstance(trend, pd.Series) or not isinstance(trend.index, pd.DatetimeIndex) or black_friday not in trend.index:
        results.append((False, "revenue_7d should be a Series with one value per day: daily[\"revenue\"].rolling(7).mean() (topic 6.2)."))
    elif abs(trend.loc[black_friday] - 24749.864286) < 0.01:
        results.append((False, "revenue_7d is centered on each day. Leave center out, so each value is that day and the six before it."))
    elif abs(trend.loc[black_friday] - 21465.394286) > 0.01 or abs(trend.loc[last_black_friday] - 14926.762857) > 0.01:
        results.append((False, f"revenue_7d should be the mean of each day and the six before it: about 21,465.39 on Black Friday 2025 and 14,926.76 on Black Friday 2024. It's {trend.loc[black_friday]:,.2f} and {trend.loc[last_black_friday]:,.2f}."))
    else:
        results.append((True, "The 7-day average of revenue on Black Friday was $21,465.39 in 2025, against $14,926.76 on Black Friday 2024."))

    bf_week = found.get("bf_week")
    last_year = found.get("last_year_week")
    change = found.get("bf_week_yoy")
    if not isinstance(bf_week, pd.DataFrame) or not isinstance(bf_week.index, pd.DatetimeIndex) or len(bf_week) == 0:
        results.append((False, "bf_week should be the daily table's rows from the Monday before BLACK_FRIDAY to the Sunday after: daily.loc[BLACK_FRIDAY - pd.Timedelta(days=4):BLACK_FRIDAY + pd.Timedelta(days=2)]."))
    elif bf_week.index[0] == pd.Timestamp("2025-11-23"):
        results.append((False, "The bf_week table starts on Sunday, November 23. The weekly table's weeks are Monday to Sunday: start on Monday, November 24."))
    elif len(bf_week) != 7 or bf_week.index[0] != pd.Timestamp("2025-11-24") or bf_week.index[-1] != week:
        results.append((False, f"The bf_week table should be the 7 days from Monday, November 24 to Sunday, November 30, 2025. It has {len(bf_week)} rows, from {bf_week.index[0]:%B %d} to {bf_week.index[-1]:%B %d}."))
    elif list(bf_week.columns) != ["sessions", "transactions", "revenue"]:
        results.append((False, f"The bf_week table should have the columns sessions, transactions and revenue. It has {list(bf_week.columns)}."))
    elif not isinstance(last_year, pd.DataFrame) or not isinstance(last_year.index, pd.DatetimeIndex) or len(last_year) == 0:
        results.append((False, "last_year_week should be the daily table's rows for the same week of 2024, with the same weekdays: 52 weeks before bf_week's days."))
    elif last_year.index[0] == pd.Timestamp("2024-11-24"):
        results.append((False, "The last_year_week table starts on November 24, 2024: a year back by the calendar, and a Sunday. Black Friday 2024 was Friday, November 29, so its week is Monday, November 25 to Sunday, December 1. Go back 52 weeks: pd.Timedelta(weeks=52)."))
    elif len(last_year) != 7 or last_year.index[0] != pd.Timestamp("2024-11-25") or list(last_year.columns) != ["sessions", "transactions", "revenue"]:
        results.append((False, f"The last_year_week table should be the 7 days from Monday, November 25 to Sunday, December 1, 2024, with the bf_week table's columns. It has {len(last_year)} rows, from {last_year.index[0]:%B %d}."))
    elif not isinstance(change, pd.Series) or not {"sessions", "transactions", "revenue"} <= set(change.index):
        results.append((False, "bf_week_yoy should be a Series with sessions, transactions and revenue: bf_week.sum() / last_year_week.sum() - 1."))
    elif abs(change["revenue"] - 0.541001) < 0.0001:
        results.append((False, "bf_week_yoy's revenue is 0.541: the change from November 24 to 30, 2024, a year back by the calendar. Compare with Black Friday 2024's week, 52 weeks back."))
    elif abs(change["revenue"] - 0.398373) < 0.0001:
        results.append((False, "bf_week_yoy's revenue is 0.398: Black Friday against Black Friday, one day. Compare the two weeks' totals."))
    elif abs(change["revenue"] - 0.471370) < 0.0001:
        results.append((False, "bf_week_yoy's revenue is 0.471: that's the whole quarter's change. Compare the two weeks' totals."))
    elif abs(change["revenue"] - 0.431242) > 0.0001 or abs(change["sessions"] - 0.405790) > 0.0001 or abs(change["transactions"] - 0.476510) > 0.0001:
        results.append((False, f"bf_week_yoy should be the bf_week table's totals over the last_year_week table's, minus 1: sessions 0.4058, transactions 0.4765 and revenue 0.4312. It is {change.round(4).to_dict()}."))
    else:
        results.append((True, "Black Friday week 2025 against the same week of 2024: sessions up 40.6%, transactions 47.7% and revenue 43.1%."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run. Read your text cell again: it's what the e-commerce team will read.")


check_your_work()

## Hints

Stuck on a step? Open its hint. Each one says where the course showed it, and gives the shape of the code, with `...` where your part goes. Try the step from the hint before you open the solution.

<details>
<summary><b>Step 1: one row per day</b></summary>

Topic 6.1.: total the `sessions` table by date, without `as_index=False`, so the dates become the index.

~~~python
daily = sessions.groupby(...)[[..., ..., ...]].sum()
daily.shape, daily.index
~~~

</details>

<details>
<summary><b>Step 2: full weeks, a year apart</b></summary>

Topic 6.2.: total by week with `resample("W")`, count each week's days, and keep the weeks with 7. Then topic 6.3.: compare each week with the row 52 weeks earlier. Because the weeks are all full and none is missing, 52 rows back is the same Monday-to-Sunday week a year earlier.

~~~python
days_per_week = daily["sessions"].resample(...).count()
weekly = daily.resample(...).sum()
weekly = weekly[days_per_week == ...].copy()
weekly["sessions_yoy"] = weekly["sessions"].pct_change(...)
weekly["revenue_yoy"] = ...
~~~

</details>

<details>
<summary><b>Step 3: the 7-day average</b></summary>

Topic 6.2.: the mean of each day and the six before it.

~~~python
revenue_7d = daily["revenue"].rolling(...).mean()
revenue_7d.loc[BLACK_FRIDAY]
~~~

</details>

<details>
<summary><b>Step 4: Black Friday week, both years</b></summary>

Topic 6.1.: a slice by dates includes both ends, and `pd.Timedelta` moves a date by days. Black Friday is a Friday, so its week starts 4 days before it and ends 2 days after. For last year's week, topic 6.3.: go back 52 weeks, so the days are the same weekdays. A year back by the calendar, `pd.DateOffset(years=1)`, lands on a Thursday.

~~~python
bf_week = daily.loc[BLACK_FRIDAY - pd.Timedelta(days=...):BLACK_FRIDAY + pd.Timedelta(days=...)]
last_year = BLACK_FRIDAY - pd.Timedelta(weeks=...)
last_year_week = daily.loc[...:...]
bf_week.index[0], bf_week.index[-1], last_year_week.index[0], last_year_week.index[-1]
~~~

</details>

<details>
<summary><b>Step 5: the change</b></summary>

Topic 6.1.: total each week first, then divide, the same way for every column at once.

~~~python
bf_week_yoy = bf_week.sum() / ... - 1
~~~

</details>

<details>
<summary><b>Step 6: the text cell</b></summary>

Choose **Insert → Text cell**. Give the week's growth, say which days of 2024 it's compared with, and why: what day of the week was Black Friday each year? Then compare the 7-day average on Black Friday 2025 with Black Friday 2024's, and the week's growth with the weekly table's other weeks.

</details>

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
daily = sessions.groupby("date")[["sessions", "transactions", "revenue"]].sum()
daily.shape
```

```python
days_per_week = daily["sessions"].resample("W").count()
weekly = daily.resample("W").sum()
weekly = weekly[days_per_week == 7].copy()
assert (weekly.index.to_series().diff().dropna() == pd.Timedelta(weeks=1)).all()
weekly["sessions_yoy"] = weekly["sessions"].pct_change(52)
weekly["revenue_yoy"] = weekly["revenue"].pct_change(52)
weekly.loc["2025-10":"2025-12"]
```

```python
revenue_7d = daily["revenue"].rolling(7).mean()
revenue_7d.loc[[BLACK_FRIDAY, BLACK_FRIDAY - pd.Timedelta(weeks=52)]]
```

```python
bf_week = daily.loc[BLACK_FRIDAY - pd.Timedelta(days=4):BLACK_FRIDAY + pd.Timedelta(days=2)]
last_year = BLACK_FRIDAY - pd.Timedelta(weeks=52)
last_year_week = daily.loc[last_year - pd.Timedelta(days=4):last_year + pd.Timedelta(days=2)]
assert list(bf_week.index.day_name()) == list(last_year_week.index.day_name())
bf_week_yoy = bf_week.sum() / last_year_week.sum() - 1
last_year.day_name(), last_year_week.index[0], last_year_week.index[-1], bf_week_yoy
```

```markdown
**For the e-commerce team.** Black Friday week 2025, Monday, November 24 to Sunday, November 30, had 64,194 sessions and $161,833.46 of revenue: up 40.6% and 43.1% on the same week of 2024, and transactions up 47.7%.

It's compared with November 25 to December 1, 2024, the week of Black Friday 2024, which was November 29. Black Friday is set by the weekday, so last year's week is 52 weeks back, Monday to Sunday. The same dates a year back, November 24 to 30, 2024, would start on a Sunday, leave out the Sunday after Black Friday, and show revenue up 54.1%: 11 points of growth that the calendar made up.

On Black Friday itself, the 7-day average of revenue was $21,465.39, against $14,926.76 on Black Friday 2024, up 43.8%. October to December as a whole grew 47.1%, so Black Friday week grew with the business, not ahead of it.
```

</details>